# Skill Memory vs. OCL Survey strategies

This notebook runs the same five Skill Memory replicates (seeds 0–4) and compares them with the available OCL Survey results.

Final accuracy is taken from the **last non-NaN test-stream accuracy for each seed**, rather than assuming the last training row contains a test-stream value.


## 0. Setup and fresh Skill Memory replicates

The experiment cell is executable so a fresh notebook run does not depend on stale Skill Memory results.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/kobros-tech/ocl_survey.git"
REPO_REF = "feature/skill-memory-comparison-notebook"
COLAB_ROOT = Path("/content/ocl_survey")

if "google.colab" in sys.modules and not (COLAB_ROOT / "experiments" / "main.py").exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)], check=True)

REPO_ROOT = COLAB_ROOT if (COLAB_ROOT / "experiments" / "main.py").exists() else Path.cwd().resolve()
if not (REPO_ROOT / "experiments" / "main.py").exists():
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "experiments" / "main.py").exists():
            REPO_ROOT = parent
            break

if not (REPO_ROOT / "experiments" / "run_skill_memory_replicates.py").exists():
    raise FileNotFoundError(f"Could not find experiments/run_skill_memory_replicates.py under {REPO_ROOT}")

print(f"Repository: {REPO_ROOT}")
print(f"Reference: {REPO_REF if 'google.colab' in sys.modules else 'local checkout'}")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT)
subprocess.run([
    sys.executable, "experiments/run_skill_memory_replicates.py"
], cwd=REPO_ROOT, env=env, check=True)


## 1. Load results

Skill Memory and ER must have exactly seeds 0–4. The notebook fails instead of silently comparing unequal replicate counts.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.toolkit.process_results import extract_results
from src.toolkit.post_metrics import compute_average_forgetting, compute_AAA, compute_wcacc

RESULTS_ROOT = REPO_ROOT / "results"
BENCHMARK = "split_cifar100"
NUM_TASKS = 20
MEMORY_SIZE = 2000
EXPECTED_SEEDS = [0, 1, 2, 3, 4]

STRATEGIES = {
    "Skill Memory": "skill_memory",
    "ER": "er",
    "ER-ACE": "er_ace",
    "DER++": "der",
    "MIR": "mir",
    "ER + LwF": "er_lwf",
    "RAR": "rar",
    "SCR": "scr",
    "AGEM": "agem",
    "MER": "mer",
    "iCaRL": "icarl",
    "GDumb": "gdumb",
}

TEST_STREAM = "Top1_Acc_Stream/eval_phase/test_stream/Task000"
VALID_STREAM = "Top1_Acc_Stream/eval_phase/valid_stream/Task000"
TEST_EXP = "Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp"

def result_path(prefix):
    return RESULTS_ROOT / f"{prefix}_{BENCHMARK}_{NUM_TASKS}_{MEMORY_SIZE}"

frames = {}
for label, prefix in STRATEGIES.items():
    path = result_path(prefix)
    if not path.is_dir():
        continue
    try:
        frames[label] = extract_results(str(path), verbose=False)
    except Exception as exc:
        warnings.warn(f"Could not load {label}: {exc}")

def seeds_for(frame):
    training = frame.get("training", pd.DataFrame())
    if "seed" not in training.columns:
        return []
    return sorted(training["seed"].dropna().astype(int).unique().tolist())

seed_table = pd.DataFrame([{"method": label, "seeds": seeds_for(frame)} for label, frame in frames.items()])
display(seed_table)

for label in ("Skill Memory", "ER"):
    if label not in frames:
        raise RuntimeError(f"{label} results are required for the comparison.")
    actual = seeds_for(frames[label])
    if actual != EXPECTED_SEEDS:
        raise RuntimeError(f"{label} has seeds {actual}; expected exactly {EXPECTED_SEEDS}.")


## 2. Scalar comparison

Final accuracy is the last **valid** test-stream accuracy recorded for each seed. Avalanche can append rows for training/evaluation events after the last test-stream value, so `seed_df.iloc[-1][TEST_STREAM]` can be NaN even though the run has a valid final test accuracy.

Forgetting is computed with the repository's causal average-forgetting implementation. AAA and WC-Acc are included only when their validation metrics contain valid values.


In [ ]:
def last_valid_metric(seed_df, metric_name):
    if metric_name not in seed_df.columns:
        return np.nan
    values = seed_df.sort_values("mb_index")[metric_name].dropna()
    return values.iloc[-1] if not values.empty else np.nan

def aggregate_seed_metric(values, label, metric_name):
    values = pd.Series(values, dtype=float)
    if values.isna().all():
        raise RuntimeError(f"{label}: no valid {metric_name} values were found for any seed.")
    if values.isna().any():
        missing = values[values.isna()].index.tolist()
        raise RuntimeError(f"{label}: missing {metric_name} for seeds {missing}.")
    return values.mean(), values.std(ddof=1) if len(values) > 1 else np.nan

summary = []

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty or TEST_STREAM not in df.columns:
        continue

    required = [f"{TEST_EXP}{i:03d}" for i in range(NUM_TASKS)]
    if not set(required).issubset(df.columns):
        warnings.warn(f"Skipping metrics for {label}: missing task accuracy columns")
        continue

    seed_values = []
    forgetting_values = []

    for seed, seed_df in df.groupby("seed"):
        seed = int(seed)
        value = last_valid_metric(seed_df, TEST_STREAM)
        if pd.isna(value):
            raise RuntimeError(f"{label}, seed {seed}: no valid final test-stream accuracy found.")
        seed_values.append(value)

        forgetting_df = compute_average_forgetting(seed_df.copy(), NUM_TASKS)
        forgetting = last_valid_metric(forgetting_df, "Average_Forgetting")
        if pd.isna(forgetting):
            raise RuntimeError(f"{label}, seed {seed}: no valid final forgetting value found.")
        forgetting_values.append(forgetting)

    final_accuracy, final_accuracy_std = aggregate_seed_metric(seed_values, label, "final accuracy")
    forgetting, forgetting_std = aggregate_seed_metric(forgetting_values, label, "forgetting")

    row = {
        "method": label,
        "n": len(seed_values),
        "final_accuracy": final_accuracy,
        "final_accuracy_std": final_accuracy_std,
        "forgetting": forgetting,
        "forgetting_std": forgetting_std,
        "AAA": np.nan,
        "AAA_std": np.nan,
        "WCAcc": np.nan,
        "WCAcc_std": np.nan,
    }

    try:
        aaa_df = compute_AAA(df.copy(), VALID_STREAM)
        aaa_values = [last_valid_metric(seed_df, "AAA") for _, seed_df in aaa_df.groupby("seed")]
        aaa_values = [v for v in aaa_values if not pd.isna(v)]
        if aaa_values:
            row["AAA"] = np.mean(aaa_values)
            row["AAA_std"] = np.std(aaa_values, ddof=1) if len(aaa_values) > 1 else np.nan
    except Exception:
        pass

    try:
        wc_df = compute_wcacc(df.copy(), NUM_TASKS)
        wc_values = [last_valid_metric(seed_df, "WCAcc") for _, seed_df in wc_df.groupby("seed")]
        wc_values = [v for v in wc_values if not pd.isna(v)]
        if wc_values:
            row["WCAcc"] = np.mean(wc_values)
            row["WCAcc_std"] = np.std(wc_values, ddof=1) if len(wc_values) > 1 else np.nan
    except Exception:
        pass

    summary.append(row)

summary_df = pd.DataFrame(summary).sort_values("final_accuracy", ascending=False)
display(summary_df.style.format({
    "final_accuracy": "{:.2%}", "final_accuracy_std": "{:.2%}",
    "forgetting": "{:.2%}", "forgetting_std": "{:.2%}",
    "AAA": "{:.2%}", "AAA_std": "{:.2%}",
    "WCAcc": "{:.2%}", "WCAcc_std": "{:.2%}",
}))

analysis_dir = RESULTS_ROOT / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(analysis_dir / "skill_memory_strategy_comparison.csv", index=False)
print(f"Saved comparison table to {analysis_dir / 'skill_memory_strategy_comparison.csv'}")


## 3. Final accuracy comparison

The plot uses the same per-seed final values used in the scalar table.


In [ ]:
plot_df = summary_df.dropna(subset=["final_accuracy"]).copy()
if plot_df.empty:
    raise RuntimeError("No final accuracy values are available to plot.")

plt.figure(figsize=(12, 5))
plt.bar(plot_df["method"], plot_df["final_accuracy"])
plt.ylabel("Final test-stream accuracy")
plt.xlabel("Method")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Metric definition

A final accuracy value is valid when the test-stream metric has at least one non-NaN observation for that seed. We select its last valid observation by `mb_index`. We do **not** convert missing metrics to zero. If a required seed has no valid final accuracy, the notebook fails explicitly.
